In [1]:
import dataikuapi
from typing import Dict, Any
import os
from sklearn.metrics.pairwise import cosine_similarity
from rapidfuzz import fuzz
import io
import fitz  # PyMuPDF
from soa_extraction.review_table import generate_review_table
import pdfplumber
from uuid import uuid4
from soa_extraction.opensearch_utils import OpensearchUtil
from soa_extraction.digitization import historical_CRF

import sys
import os
import fitz  # PyMuPDF
# from langgraph_utils.Info_extractor import InfoExtractorAgent
# from langgraph_utils.chat_history import SnowflakeChatMessageHistory
import json
# from langgraph_utils.file_parser import FileParser
# from langgraph_utils.digitization import main_handler
import uuid
# from langgraph_utils import creds
# from langgraph_utils.variables import PROJECT_NAME, SECRET_NAME, TOKEN_KEY
from utils.connection import get_dataiku_client_and_project
import logging



DATAIKU_HOST = "http://10.45.152.66:10000"
API_SECRET_KEY = "dkuaps-b3EsRXVjU3w4y7nd4KEwibEr04CjFPZr"          
PROJECT_NAME = "ECSGENERATION"   

# client, proj = get_dataiku_client_and_project(PROJECT_NAME, SECRET_NAME, TOKEN_KEY)
client = dataikuapi.DSSClient(DATAIKU_HOST, API_SECRET_KEY)
proj = client.get_project(PROJECT_NAME)

{'type': 'ElasticSearch', 'params': {'host': 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com', 'username': 'genai-admin', 'password': 'qPk7Jf5vcyXDS!**332gXTvSfmcauvr9', 'port': 443, 'ssl': True, 'trustAnySSLCertificate': True, 'dialect': 'ES_7', 'dkuProperties': [], 'namingRule': {'indexNameDatasetNamePrefix': '${projectKey}_'}, 'authType': 'PASSWORD', 'oauth': {'refreshTokenRotation': False}, 'aws': {'service': 'OPENSEARCH_SERVERLESS', 'credentialsMode': 'KEYPAIR', 'customAWSCredentialsProviderParams': []}}, 'credentialsMode': 'GLOBAL', 'proxySettingsAsString': ''}
opensearch <OpenSearch([{'host': 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com', 'port': 443}])>
ecsgeneration_ecs_index_data


/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/opensearchpy/connection/http_urllib3.py:214: UserWarning: Connecting to https://aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com:443 using SSL with verify_certs=False is insecure.
  warnings.warn(


In [40]:
import os
import io
import fitz  # PyMuPDF
import pdfplumber
from uuid import uuid4


class historical_CRF:
    
    def __init__(self, client, proj, chunk_size=1000):
        self.proj = proj
        self.client = client
        self.s3_folder_dataset_id = proj.get_variables()['local'].get('file_upload') # change file upload 
        self.input_folder = proj.get_managed_folder(self.s3_folder_dataset_id)
        self.files = self.input_folder.list_contents()["items"]
        self.toc_page_limit = 20
        self.config = proj.get_variables()["local"]
        print(self.files)

    def historical_mapping(self, file_path,file_id,paths=[]):
        result = []

#         for file in paths:
#             path = file
#             print(path)
#             parts = path.strip('/').split('/')

#             if "Historical" in path:
                
#                 result.append({
                    
#                     "path": path,
                   
    
#                 })
        result.append(file_path)

        response = []
        snowflake_conn = self.config.get("snowflake_connection_string")
        table_file_upload = self.config.get("ecs", {}).get("ecs_file_upload", {})

        for i in result:
            

            with self.input_folder.get_file(file_path) as stream:
                file_bytes = stream.raw.data
            
            
            try:
                
                import re

                def clean_summary(text):
                    # Remove newlines, tabs, and collapse extra spaces
                    text = re.sub(r'\s+', ' ', text).strip()

                    # Ensure it ends with a single period
                    if not text.endswith('.'):
                        text += '.'

                    return text
                
                pdf_file_like = io.BytesIO(file_bytes)
                with pdfplumber.open(pdf_file_like) as pdf:
                    total_pages = len(pdf.pages)
                    print(total_pages)
                    snowflake_conn = self.config.get("snowflake_connection_string")
                    table_file_upload = self.config.get("ecs", {}).get("ecs_file_upload", {})
                    
                    for page_num, page in enumerate(pdf.pages):
                        res = {}
                        percent = (page_num / total_pages) * 100
                        
                        update_query = f"""
                                        UPDATE {table_file_upload}
                                        SET "digitization_percent" = '{percent}'
                                        WHERE "crf_file_id" = '{file_id}'
                                         ;
                                    """
                        client.sql_query(query = update_query,connection = snowflake_conn,  post_queries=["COMMIT"])
#                         parts = i["path"].strip('/').split('/')
                        therapeutic_area = ''
                        source =  "Unknown"
                        template_name = os.path.basename(file_path)
                        unique_id = uuid4()
#                         print(hist_id)
                        res = {
                           
                            
                            "template_name": template_name,
                            "path": file_path,
                            "id": unique_id,

                        }
                        
                        page_text = page.extract_text(layout=True)
                        
                        if page_text and "field name" in page_text.lower():
                            continue

                        if not page_text:
                            continue

                        lines = page_text.split('\n')
                        header_lines = []
                        field_value_map = {}
                        current_field = ""
                        in_field_section = False

                        for line in lines:
                            line = line.strip()
                            if not line:
                                continue

                            # Trigger point for header vs fields
                            if not in_field_section:
                                if "generated" in line.lower():
                                    in_field_section = True
                                    continue
                                header_lines.append(line)
                            else:
                                if len(line.strip()) == 0:
                                    continue
                                
                                pattern = r'''
                                    ^                              # Start of line
                                    (?P<field>.+?)                 # Field name (non-greedy)
                                    (?:\t|\s{2,})+                 # Separator: tab or ≥2 spaces
                                    (?P<value>.+?)                 # Value
                                    \s*$                           # Optional trailing spaces
                                '''
                                pattern2  = r'^(?!\s)(?!.*\s$)(?P<value>.+)$'

                                
                                
                                
                                
                                
#                                 current_field = None

                                # Step 1 ─ collect every “proper” field line
                                m = re.match(pattern, line, re.VERBOSE)
                                p = re.match(pattern2, line, re.VERBOSE) 
                                if m:
                                    if m.group("field") and m.group("value"):
                                        feild = m.group("field")
                                        current_field = feild
                                        value = m.group("value")
                                        
                                        
                                if p:
                                    if p.group("value") and current_field:
                                        # Continuation of previous field
#                                         feild = current_field
                                        value = p.group("value")
                                        
                        
                               
                                
                                left_part = current_field.strip()
                                right_part = value.strip()

                                if left_part:
                                    if left_part in field_value_map and right_part:
                                        field_value_map[left_part].append(right_part)
                                     
                                    else:
                                        
                                        field_value_map[left_part] = [right_part.strip()]
                             

                        final_feilds = []
                        for k in field_value_map:
                            
                            final_feilds.append({
                                "field_name" : k,
                                "field_value": field_value_map[k]
                            })
                        
                        head = ""
                        for j in header_lines:
                            if "form" in j.lower().strip() or "folder" in j.lower().strip():
                                head += j + " "
                        
                        res["source_data"] = {
                            "assessments" : head,
                            "fields" : final_feilds 
                            
                        }
#                         print(res)
                    
#                     print(res)
                        if final_feilds and head:
                            response.append(res)
#                             print(f"✅ extraction for {i['path']} completed")
                    

            except Exception as e:
                print(f"Error processing file {i['path']}: {e}")
        
        

        return response


NameError: name 'dataiku' is not defined

In [6]:
def llm_similarity_score(field_1, field_2):
    prompt = f"""Your job is to get the similarity between 2 sentences , provide the similariy between them , dont explain the answer just provide the number 
    
    first sentence : {field_1}
    second sentence : {field_2}
    """
    default_llm_model = proj.get_variables()['local'].get('default_llm_model')
        #self.default_llm_model = 'azureopenai:Azure-OpenAi:gpt-4o'
#         logging.info(f"[PlannerAgent] Initialized with LLM model: {self.default_llm_model}")
       
    llm = proj.get_llm(default_llm_model).as_langchain_llm(
        completion_settings={
        "temperature": 0,
                
                "timeout": 300,
            "max_tokens": 8192
            # 5 minutes
            })
    output = llm.invoke(prompt)
        
    return output

In [7]:
float(llm_similarity_score('Informed consent date','Date of Consent'))

/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/pydantic/_internal/_config.py:341: UserWarning: Valid config keys have changed in V2:
* 'underscore_attrs_are_private' has been removed
  warnings.warn(message, UserWarning)


0.85

In [9]:
#necessary imports 
from datetime import datetime
import traceback
import dataikuapi
import io

import base64
import uuid
# import from GLOBAL SHARED CODE
from utils import connection 
# imports from library 
import re
from utilities.variables import RD_PROJECT_NAME
from utilities.variables import SECRET_NAME , TOKEN_KEY
from utilities.logging_config import logging
# from soa_extraction.crf_extraction import HistoricalCRF
import traceback
import os


def extract_crf(file_path,file_id,created_by):
    """
    Input Args:
        file_path: str
        
    Response:
        result : dict
    """
    try:
        logging.info("Intializing client for file_upload function ")
       
        DATAIKU_HOST , API_SECRET_KEY  = connection.get_dataiku_host_and_api_key(RD_PROJECT_NAME,SECRET_NAME,TOKEN_KEY)
        
        client = dataikuapi.DSSClient(DATAIKU_HOST, API_SECRET_KEY)
        project = client.get_project(RD_PROJECT_NAME)
        proj_vars = project.get_variables()["local"]
        crf_output_table = proj_vars.get('ctl_crf_output')
        snowflake_conn = proj_vars.get("snowflake_connection_string")
        
        obj = historical_CRF(client , project)
        final_list = []
        response = obj.historical_mapping(file_path,file_id)
        print(response)
        for resp in response:
            form_name = resp['source_data']['assessments']
            print(form_name)
            match = re.search(r'Form[:\s]*(.*)', form_name)
            if match:
                form_name = match.group(1)
            for field in resp['source_data']['fields']:
                field_name = field['field_name']
                final_list.append({
                    "form_name": form_name,
                    "field_name": field_name
                })

        # ✅ Deduplicate based on both form_name and field_name
        seen = set()
        deduped_list = []
        for item in final_list:
            key = (item["form_name"].strip().lower(), item["field_name"].strip().lower())
            if key not in seen:
                seen.add(key)
                deduped_list.append(item)
                
        string_json = json.dumps(deduped_list)

        # Insert into Snowflake
        insert_query = f"""
        INSERT INTO {crf_output_table} ("crf_file_id", "created_by", "crf_output")
        VALUES ('{file_id}', '{created_by}', $$ {string_json} $$)
        """
        client.sql_query(query=insert_query, connection=snowflake_conn, post_queries=["COMMIT"])

        return {
            "result": deduped_list
        }

    except:
        t = traceback.format_exc()
        logging.error(f"Error caused due to {t}")
        return {"message": f"Error caused due to {t}"}


In [10]:
# from 
output_crfs = extract_crf('/Annotated_Otsuka_405 201 00157_00150405_v1.0_Complete eCRF (1).pdf'
                          ,'c0cd4597-b3fa-479b-a0ad-f6b81f5eb594','')

[{'path': '/3', 'size': 630, 'lastModified': 1758528240000}, {'path': '/323-201-00002Protocol.docx', 'size': 186925, 'lastModified': 1759820250000}, {'path': '/Annotated_Otsuka_405 201 00157_00150405_v1.0_Complete eCRF (1) (4).pdf', 'size': 2544020, 'lastModified': 1760616641000}, {'path': '/Annotated_Otsuka_405 201 00157_00150405_v1.0_Complete eCRF (1).pdf', 'size': 2544020, 'lastModified': 1757496080000}, {'path': '/ECS_output.xlsx', 'size': 2635330, 'lastModified': 1759156409000}, {'path': '/output.csv', 'size': 102306, 'lastModified': 1760369801000}, {'path': '/output.xlsx', 'size': 57081, 'lastModified': 1760337547000}, {'path': '/soa_1.pdf', 'size': 1178968, 'lastModified': 1759826677000}, {'path': '/soatest1.pdf', 'size': 1178968, 'lastModified': 1759763806000}, {'path': '/soatest2.pdf', 'size': 1178968, 'lastModified': 1759819989000}, {'path': '/standardECS.xlsx', 'size': 267281, 'lastModified': 1758524961000}, {'path': '/standard_ECS_output.xlsx', 'size': 267282, 'lastModified

In [11]:
(output_crfs['result'])

[{'form_name': 'Enrollment ', 'field_name': 'Site ID'},
 {'form_name': 'Enrollment ', 'field_name': 'Participant ID'},
 {'form_name': 'Enrollment ', 'field_name': 'Participant Number (Derived)'},
 {'form_name': 'Date of Visit ', 'field_name': 'Visit date'},
 {'form_name': 'Informed Consent ',
  'field_name': 'Informed consent obtained?'},
 {'form_name': 'Informed Consent ', 'field_name': 'Informed consent date'},
 {'form_name': 'Informed Consent ', 'field_name': 'Informed consent time'},
 {'form_name': 'Informed Consent ', 'field_name': 'Derived date'},
 {'form_name': 'Informed Consent ',
  'field_name': 'Informed consent version number'},
 {'form_name': 'Informed Consent ',
  'field_name': 'Standardized disposition term'},
 {'form_name': 'Informed Consent ', 'field_name': 'Protocol version'},
 {'form_name': 'Demographics ', 'field_name': 'Date of Birth'},
 {'form_name': 'Demographics ', 'field_name': 'Age'},
 {'form_name': 'Demographics ', 'field_name': 'Sex at Birth'},
 {'form_name':

In [12]:
# output_crfs = extract_crf('/Annotated_Otsuka_405 201 00157_00150405_v1.0_Complete eCRF (1).pdf')
# Single search over OpenSearch index with hybrid similarity (vector + fuzzy)
import time
from sklearn.metrics.pairwise import cosine_similarity
from rapidfuzz import fuzz  # for fuzzy text similarity

start = time.time()

# Initialize OpenSearch client
opensearch_client = OpensearchUtil(client, proj)
client_os = opensearch_client.opensearch_client

output_list = []

# Get the OpenSearch index name from project variables
index_name = proj.get_variables()['local'].get('ecs_opensearch')
standard_index = "${projectKey}_ecs_index_data"
dataiku_project_var = "${projectKey}"
if dataiku_project_var in index_name:
    # Replace projectKey placeholder with actual project key
    index_name = index_name.replace(dataiku_project_var, opensearch_client.project.project_key).lower()
    standard_index = standard_index.replace(dataiku_project_var, opensearch_client.project.project_key).lower()
    print(standard_index)

{'type': 'ElasticSearch', 'params': {'host': 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com', 'username': 'genai-admin', 'password': 'qPk7Jf5vcyXDS!**332gXTvSfmcauvr9', 'port': 443, 'ssl': True, 'trustAnySSLCertificate': True, 'dialect': 'ES_7', 'dkuProperties': [], 'namingRule': {'indexNameDatasetNamePrefix': '${projectKey}_'}, 'authType': 'PASSWORD', 'oauth': {'refreshTokenRotation': False}, 'aws': {'service': 'OPENSEARCH_SERVERLESS', 'credentialsMode': 'KEYPAIR', 'customAWSCredentialsProviderParams': []}}, 'credentialsMode': 'GLOBAL', 'proxySettingsAsString': ''}
opensearch <OpenSearch([{'host': 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com', 'port': 443}])>
ecsgeneration_ecs_index_data


/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/opensearchpy/connection/http_urllib3.py:214: UserWarning: Connecting to https://aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com:443 using SSL with verify_certs=False is insecure.
  warnings.warn(


In [5]:
import pandas as pd
final_df = pd.DataFrame(output_crfs['result'])
 w_cos * cos_sim + w_text * text_score

In [21]:
i = 0
output_list = []

def compute_similarity(vec1, vec2, text1, text2, w_cos=0.8, w_text=0.2):
    cos_sim = cosine_similarity([vec1], [vec2])[0][0]
    print('cosine score ->',cos_sim)
    text_score = fuzz.token_sort_ratio(text1, text2) / 100
    print('cosine score ->',cos_sim,'=fuzz score=',text_score)
    return cos_sim


for row in output_crfs['result']:
    i += 1
    form_name, field_name = row['form_name'], row['field_name']
    print(form_name, field_name )

    # Embeddings
    form_vec = opensearch_client.create_embedding(
        form_name, proj.get_variables()['local'].get("default_embeddings_model_id")
    )['response']
    field_vec = opensearch_client.create_embedding(
        field_name, proj.get_variables()['local'].get("default_embeddings_model_id")
    )['response']

    # -------- Historic Search -------- #
    hist_query = {
        "query": {
            "bool": {
                "should": [
                    {"knn": {"form_name_vector": {"vector": form_vec, "k": 10}}},
                    {"knn": {"form_field_value_vector": {"vector": field_vec, "k": 10}}}
                ],
                "filter": [{"term": {"source.keyword": "Historic"}}]
            }
        }
    }

    hist_result = opensearch_client.opensearch_client.search(
        index=index_name, body=hist_query, size=5
    )

    best_hist = None
    best_hist_score = -1
    for hit in hist_result["hits"]["hits"]:
        if "form_field_value_vector" in hit["_source"]:
            score = compute_similarity(
                field_vec, hit["_source"]["form_field_value_vector"],
                field_name, hit["_source"]["form_field_value"]
            )
            print("historic score",hit['_score'])
#             score = float(llm_similarity_score( field_name, hit["_source"]["form_field_value"]))
        else:
            continue
            score = fuzz.token_sort_ratio(
                form_name.lower().strip(),
                hit["_source"]["form_name"].lower().strip()
            ) / 100.0

        if score > best_hist_score:
            best_hist_score, best_hist = score, hit

    if best_hist and best_hist_score > 0.65:
        best_hist['_source'].update({
            "original_form_name": form_name,
            "original_field": field_name,
            "score": best_hist['_score'] / 2,
            "fuzzy_or_vector_score": best_hist_score
        })
        output_list.append(best_hist['_source'])
        continue  # ✅ Found in Historic, skip Standard

    # -------- Standard Search (using compute_similarity) -------- #
    print("Switching to Standard…")

    std_query = {
        "query": {
            "bool": {
                "should": [
                    {"knn": {"form_name_vector": {"vector": form_vec, "k": 10}}},
                    {"knn": {"form_field_vector": {"vector": field_vec, "k": 10}}}
                ],
                "filter": [{"term": {"source.keyword": "Standard"}}]
            }
        }
    }

    std_result = opensearch_client.opensearch_client.search(
        index=standard_index, body=std_query, size=5
    )

    best_std = None
    best_score = -1
    
    print('====original field name=====',field_name)
    for hit in std_result["hits"]["hits"]:
       

        # 🔑 Compute hybrid similarity instead of pure fuzzy
        score = compute_similarity(
            field_vec, hit["_source"]["form_field_vector"],
            field_name, hit["_source"]["form_field_value"]
        )
#         score = float(llm_similarity_score( field_name, hit["_source"]["form_field_value"]))
        print('----------------------',score,'opensearch field', hit["_source"]["form_field_value"])

        if score > best_score:
            best_score = score
            best_std = hit

    # --- Post-filtering & LLM fallback --- #
    print('total score',best_std['_score'])
    if best_std:
        best_std['_source'].update({
            "original_form_name": form_name,
            "original_field": field_name,
            "score": best_std['_score'] / 2,
            "fuzzy_or_vector_score": best_score
        })

        if form_name.lower().strip() == "informed consent":
            print(f"original: {form_name}, {field_name}")
            print(f"hit: {best_std['_source']['form_name']}, {best_std['_source']['form_field_value']}")
            print(f"similarity_score={best_score:.3f}, opensearch_score={best_std['_score']/2:.3f}")

        # ✅ Accept if hybrid score passes threshold
        if best_std['_score'] / 2 > 0.64 and best_score > 0.60:
            print('appened')
            output_list.append(best_std['_source'])
        else:
            # ❌ Too weak → fallback to LLM
            print("LLM fallback triggered…")
            output = {}  # output = json.loads(make_llm_call(form_name, field_name))
            if isinstance(output, list):
                output = output[0]

            output.update({
                "ecs_id": best_std['_source']['ecs_id'],
                "form_id": best_std['_source']['form_id'],
                "original_form_name": form_name,
                "original_field": field_name,
                "score": best_std['_score'] / 2,
                "source": "LLM Generated"
            })
            output_list.append(output)
                
        if i == 200:
            break


Enrollment  Site ID
cosine score -> 0.26749449803417163
cosine score -> 0.26749449803417163 =fuzz score= 0.2142857142857143
historic score 1.2222216
cosine score -> 0.26463345880958594
cosine score -> 0.26463345880958594 =fuzz score= 0.18867924528301885
historic score 1.2008284
cosine score -> 0.232457626070009
cosine score -> 0.232457626070009 =fuzz score= 0.19999999999999996
historic score 1.1807888
cosine score -> 0.25716142878648907
cosine score -> 0.25716142878648907 =fuzz score= 0.2857142857142857
historic score 1.1750968
cosine score -> 0.25716142878648907
cosine score -> 0.25716142878648907 =fuzz score= 0.2857142857142857
historic score 1.1750968
Switching to Standard…
====original field name===== Site ID
cosine score -> 0.4025437528860596
cosine score -> 0.4025437528860596 =fuzz score= 0.4705882352941176
---------------------- 0.4025437528860596 opensearch field Visit Date
cosine score -> 0.4025437528860596
cosine score -> 0.4025437528860596 =fuzz score= 0.4705882352941176
---

/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made t

====original field name===== Participant ID
cosine score -> 0.4189870201705205
cosine score -> 0.4189870201705205 =fuzz score= 0.2727272727272727
---------------------- 0.4189870201705205 opensearch field Age Unit
cosine score -> 0.39157054835682503
cosine score -> 0.39157054835682503 =fuzz score= 0.27586206896551724
---------------------- 0.39157054835682503 opensearch field Collection Date
cosine score -> 0.39157054835682503
cosine score -> 0.39157054835682503 =fuzz score= 0.27586206896551724
---------------------- 0.39157054835682503 opensearch field Collection Date
cosine score -> 0.39157054835682503
cosine score -> 0.39157054835682503 =fuzz score= 0.27586206896551724
---------------------- 0.39157054835682503 opensearch field Collection Date
cosine score -> 0.3377524725379938
cosine score -> 0.3377524725379938 =fuzz score= 0.5
---------------------- 0.3377524725379938 opensearch field Visit Date
total score 1.2475374
LLM fallback triggered…
Enrollment  Participant Number (Derived)

/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made t

cosine score -> 0.9028199691434449
cosine score -> 0.9028199691434449 =fuzz score= 0.5846153846153845
historic score 1.7369345
cosine score -> 0.7324005074980706
cosine score -> 0.7324005074980706 =fuzz score= 0.36144578313253023
historic score 1.6143998
cosine score -> 0.5722811486105295
cosine score -> 0.5722811486105295 =fuzz score= 0.4444444444444444
historic score 1.3052979
cosine score -> 0.5406837296353035
cosine score -> 0.5406837296353035 =fuzz score= 0.3908045977011494
historic score 1.2927971
cosine score -> 0.515106128492731
cosine score -> 0.515106128492731 =fuzz score= 0.40625
historic score 1.2865987
Informed Consent  Informed consent date
cosine score -> 0.7067116817953552
cosine score -> 0.7067116817953552 =fuzz score= 0.4666666666666666
historic score 1.5987298
cosine score -> 0.5761994032678741
cosine score -> 0.5761994032678741 =fuzz score= 0.3589743589743589
historic score 1.5278525
cosine score -> 0.5633520057363859
cosine score -> 0.5633520057363859 =fuzz score= 

...

====original field name===== If other, specify
cosine score -> 0.5803198060031483
cosine score -> 0.5803198060031483 =fuzz score= 0.368421052631579
---------------------- 0.5803198060031483 opensearch field Specify other contact
cosine score -> 0.5803198060031483
cosine score -> 0.5803198060031483 =fuzz score= 0.368421052631579
---------------------- 0.5803198060031483 opensearch field Specify other contact
cosine score -> 0.3562916886316349
cosine score -> 0.3562916886316349 =fuzz score= 0.29850746268656714
---------------------- 0.3562916886316349 opensearch field If no, what was the reason the visit was not done?
cosine score -> 0.29413912282200483
cosine score -> 0.29413912282200483 =fuzz score= 0.16666666666666663
---------------------- 0.29413912282200483 opensearch field Ongoing
cosine score -> 0.2943210013683231
cosine score -> 0.2943210013683231 =fuzz score= 0.375
---------------------- 0.2943210013683231 opensearch field Type of Consent
total score 1.2946503
LLM fallback trig

/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made t

cosine score -> 0.3266192738360665
cosine score -> 0.3266192738360665 =fuzz score= 0.34375
historic score 1.2511655
cosine score -> 0.31918320218831997
cosine score -> 0.31918320218831997 =fuzz score= 0.34782608695652173
historic score 1.2485217
cosine score -> 0.3176729271405327
cosine score -> 0.3176729271405327 =fuzz score= 0.36923076923076925
historic score 1.2479876
cosine score -> 0.26595227074437133
cosine score -> 0.26595227074437133 =fuzz score= 0.43333333333333335
historic score 1.2192615
cosine score -> 0.25608954563420616
cosine score -> 0.25608954563420616 =fuzz score= 0.33333333333333337
historic score 1.2160001
Switching to Standard…
====original field name===== Is participant a screen failure?
cosine score -> 0.28348215117795605
cosine score -> 0.28348215117795605 =fuzz score= 0.33766233766233766
---------------------- 0.28348215117795605 opensearch field Date of contact/Date of final contact attempt
cosine score -> 0.26994926065433145
cosine score -> 0.2699492606543314

/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made t

====original field name===== Screening completed or reason for screening failure
cosine score -> 0.3886744936290563
cosine score -> 0.3886744936290563 =fuzz score= 0.32432432432432434
---------------------- 0.3886744936290563 opensearch field Has the subject had any adverse events since the last visit?
cosine score -> 0.3886744936290563
cosine score -> 0.3886744936290563 =fuzz score= 0.32432432432432434
---------------------- 0.3886744936290563 opensearch field Has the subject had any adverse events since the last visit?
cosine score -> 0.3886744936290563
cosine score -> 0.3886744936290563 =fuzz score= 0.32432432432432434
---------------------- 0.3886744936290563 opensearch field Has the subject had any adverse events since the last visit?
cosine score -> 0.3584792716236339
cosine score -> 0.3584792716236339 =fuzz score= 0.33333333333333337
---------------------- 0.3584792716236339 opensearch field Date of contact/Date of final contact attempt
cosine score -> 0.3451192650020968
cosine 

In [22]:
import pandas as pd
output = pd.DataFrame(output_list)
len(output_list)

200

In [23]:
out_final = output[['original_form_name', 'original_field','source','form_name',  'form_field_value','score', 'form_domain_name','variable_name',
       'validation_logic', 'reasoning', 'action', 'action_details',]]

In [24]:
print(len(out_final))
out_final



200


,original_form_name,original_field,source,form_name,form_field_value,score,form_domain_name,variable_name,validation_logic,reasoning,action,action_details
0,Enrollment,Site ID,LLM Generated,NaN,NaN,0.606038,NaN,NaN,NaN,NaN,NaN,NaN
1,Enrollment,Participant ID,LLM Generated,NaN,NaN,0.623769,NaN,NaN,NaN,NaN,NaN,NaN
2,Enrollment,Participant Number (Derived),LLM Generated,NaN,NaN,0.622374,NaN,NaN,NaN,NaN,NaN,NaN
3,Date of Visit,Visit date,Standard,Subject Visits,Visit Date,0.792454,SV,VISDAT,(SV.VISDAT < DS_IC.DSSTDAT_IC) where (DS_IC.DS...,no subject visit date should ever be prior to ...,prompt user with ACTION DETAILS,Visit date is before the date of study partici...
4,Informed Consent,Informed consent obtained?,Historic,Subject Informed Consent,Has the informed consent been obtained?,0.868467,NaN,TEMPLATE_ICF_CONSENT_OBTAINED,Value should = Y,NaN,Must Equal Y,"Informed Consent must be obtained, please review."
...,...,...,...,...,...,...,...,...,...,...,...,...
195,Laboratory Test Collections,Repeat performed?,LLM Generated,NaN,NaN,0.598040,NaN,NaN,NaN,NaN,NaN,NaN
196,Screening DM1 Outcome,Is participant a screen failure?,LLM Generated,NaN,NaN,0.607149,NaN,NaN,NaN,NaN,NaN,NaN
197,Screening DM1 Outcome,Date screening completed/failed,LLM Generated,NaN,NaN,0.668662,NaN,NaN,NaN,NaN,NaN,NaN
198,Screening DM1 Outcome,Screening completed or reason for screening fa...,LLM Generated,NaN,NaN,0.634224,NaN,NaN,NaN,NaN,NaN,NaN


In [81]:
final_df[final_df['form_name'] == 'Weight/Height/BMI']

,form_name,field_name
67,Weight/Height/BMI,Were measurements performed?
68,Weight/Height/BMI,"If no, specify"
69,Weight/Height/BMI,Date
70,Weight/Height/BMI,Time
71,Weight/Height/BMI,Derived date
72,Weight/Height/BMI,Height
73,Weight/Height/BMI,Weight
74,Weight/Height/BMI,BMI


In [80]:
final_df

,form_name,field_name
0,Enrollment,Site ID
1,Enrollment,Participant ID
2,Enrollment,Participant Number (Derived)
3,Date of Visit,Visit date
4,Informed Consent,Informed consent obtained?
...,...,...
890,Columbia Suicide Severity Rating Scale (C-SSRS...,Preparatory Acts or Behavior:
891,Columbia Suicide Severity Rating Scale (C-SSRS...,Suicidal behavior was present during the asses...
892,Columbia Suicide Severity Rating Scale (C-SSRS...,Completed Suicide:
893,Columbia Suicide Severity Rating Scale (C-SSRS...,Most Lethal Attempt Date (dd MMM yyyy)


In [14]:
len(output_crfs['result'])

895

In [5]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import json
import re
import traceback
import dataikuapi
from utilities.logging_config import logging
from soa_extraction.crf_extraction import HistoricalCRF
from utils import connection
from utilities.variables import RD_PROJECT_NAME, SECRET_NAME, TOKEN_KEY

def extract_crf(file_path, file_id, created_by, max_workers=5):
    """
    Extract CRF fields and insert into Snowflake using threads for speed.

    Args:
        file_path: str
        file_id: str
        created_by: str
        max_workers: int, number of threads for processing

    Returns:
        dict: deduplicated CRF fields
    """
    try:
        logging.info("Initializing client for file_upload function")

        # Setup Dataiku client
        DATAIKU_HOST, API_SECRET_KEY = connection.get_dataiku_host_and_api_key(
            RD_PROJECT_NAME, SECRET_NAME, TOKEN_KEY
        )
        client = dataikuapi.DSSClient(DATAIKU_HOST, API_SECRET_KEY)
        project = client.get_project(RD_PROJECT_NAME)
        proj_vars = project.get_variables()["local"]
        crf_output_table = proj_vars.get('ctl_crf_output')
        snowflake_conn = proj_vars.get("snowflake_connection_string")

        # Extract historical CRF
        obj = HistoricalCRF(client, project)
        response = obj.historical_mapping(file_path,file_id)

        final_list = []

        # Function to process each response item
        def process_resp(resp):
            local_list = []
            form_name = resp['source_data']['assessments']
            match = re.search(r'Form[:\s]*(.*)', form_name)
            if match:
                form_name = match.group(1)
            for field in resp['source_data']['fields']:
                field_name = field['field_name']
                local_list.append({
                    "form_name": form_name,
                    "field_name": field_name
                })
            return local_list

        # Use ThreadPoolExecutor to process in parallel
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = [executor.submit(process_resp, resp) for resp in response]
            for future in as_completed(futures):
                final_list.extend(future.result())

        # Deduplicate
        seen = set()
        deduped_list = []
        for item in final_list:
            key = (item["form_name"].strip().lower(), item["field_name"].strip().lower())
            if key not in seen:
                seen.add(key)
                deduped_list.append(item)

        string_json = json.dumps(deduped_list)

        # Insert into Snowflake
        insert_query = f"""
        INSERT INTO {crf_output_table} ("crf_file_id", "created_by", "crf_output")
        VALUES ('{file_id}', '{created_by}', $$ {string_json} $$)
        """
        client.sql_query(query=insert_query, connection=snowflake_conn, post_queries=["COMMIT"])

        return {"result": deduped_list}

    except Exception:
        t = traceback.format_exc()
        logging.error(f"Error caused due to {t}")
        return {"message": f"Error caused due to {t}"}


In [14]:
output_crfs

{'result': []}